# Guided EDA — competition AMEX mart (slices 1–2)

**Collaborative session cut** — load → drop high-null columns → numeric vs categorical → class imbalance → **univariate vs target (step 5)**.

Out of this notebook (for later): correlation matrix / multicollinearity, outlier treatment, WOE/IV, modeling.

**Disclaimer:** public demo data only (not live bank BFSI).

> **Warning — book-sample mart ≠ competition prepare contract.**  
> `data/credit/application_mart/application_mart.parquet` comes from `build-application-mart` on the book pickle. Most AMEX category codes stay as floats there, so a dtype-only inventory under-counts categoricals (often only `D_63_last` / `D_64_last`).  
> This guided session loads the **competition AMEX mart** (`build-competition-mart --source-kind amex`) where official categoricals are aggregated as `{col}_mode` / `{col}_last` per [#157](https://github.com/JLaborda/SmartWealthAI/issues/157).

| Column | Meaning |
| --- | --- |
| `customer_ID` | Application / customer id (not a feature) |
| `target` | `1` = bad (default), `0` = good |


## 1. Load data

Resolution order for **this guided session**:

1. `data/credit/application_mart/amex/application_mart.parquet` (**preferred** — competition AMEX mart, `#157` prepare contract)
2. Else `data/credit/application_mart/application_mart.parquet` (book-sample mart — **warn**: wrong categorical contract)
3. Else `credit/data/train_df_sample.pkl` (Git LFS book sample)

Rebuild competition mart when raw is present:

```bash
poetry run credit-css build-competition-mart \
  --source-kind amex \
  --raw-dir data/credit/raw/amex \
  --output-dir data/credit/application_mart/amex
```

In [ ]:
from pathlib import Path

import pandas as pd

ID_COL = "customer_ID"
TARGET = "target"
NULL_DROP_THRESHOLD = 0.80

# Official AMEX categoricals (competition docs / #157) → mart names {col}_mode / {col}_last
AMEX_CATEGORICALS = [
    "B_30",
    "B_38",
    "D_114",
    "D_116",
    "D_117",
    "D_120",
    "D_126",
    "D_63",
    "D_64",
    "D_66",
    "D_68",
]
AMEX_CAT_FEATURE_NAMES = {f"{c}_{suf}" for c in AMEX_CATEGORICALS for suf in ("mode", "last")}

REPO_ROOT_CANDIDATES = [Path("../.."), Path(".")]


def resolve_repo_root() -> Path:
    for root in REPO_ROOT_CANDIDATES:
        if (root / "credit" / "notebooks").exists() or (root / "pyproject.toml").exists():
            return root.resolve()
    return Path("../..").resolve()


ROOT = resolve_repo_root()
COMPETITION_AMEX_MART = (
    ROOT / "data" / "credit" / "application_mart" / "amex" / "application_mart.parquet"
)
BOOK_MART = ROOT / "data" / "credit" / "application_mart" / "application_mart.parquet"
SAMPLE_PKL = ROOT / "credit" / "data" / "train_df_sample.pkl"

if COMPETITION_AMEX_MART.exists():
    source_path = COMPETITION_AMEX_MART
    source_label = "competition AMEX mart (preferred for this guided session)"
    df = pd.read_parquet(source_path)
elif BOOK_MART.exists():
    source_path = BOOK_MART
    source_label = (
        "book-sample application mart (FALLBACK — wrong categorical contract; "
        "rebuild competition mart)"
    )
    df = pd.read_parquet(source_path)
elif SAMPLE_PKL.exists() and SAMPLE_PKL.stat().st_size > 1_000_000:
    source_path = SAMPLE_PKL
    source_label = "book AMEX sample pickle (fallback — not the competition mart)"
    df = pd.read_pickle(source_path)
else:
    raise FileNotFoundError(
        "No usable mart/sample found.\n"
        f"  competition: {COMPETITION_AMEX_MART}\n"
        f"  book mart:   {BOOK_MART}\n"
        f"  pkl:         {SAMPLE_PKL}\n"
        'Run build-competition-mart or git lfs pull --include="credit/data/*"'
    )

print(f"source: {source_label}")
print(f"path:   {source_path}")
print(f"shape:  {df.shape[0]:,} rows × {df.shape[1]:,} columns")
print(f"id col present: {ID_COL in df.columns}; target present: {TARGET in df.columns}")
df.head(3)

## 2. Drop columns with ≥80% nulls

High missingness columns add little signal for a first pass and inflate the feature list.
We compute the null rate per column, drop those at or above **80%**, and keep a cleaned frame `df_clean`.

Id and target are never dropped here (they are required keys).

In [ ]:
rows_before, cols_before = df.shape
null_rate = df.isna().mean().sort_values(ascending=False)

print("Null-rate summary (all columns):")
print(null_rate.describe().to_string())
print()
print("Top 15 null rates:")
print(null_rate.head(15).to_string())

drop_cols = [
    c for c in null_rate.index[null_rate >= NULL_DROP_THRESHOLD] if c not in {ID_COL, TARGET}
]
df_clean = df.drop(columns=drop_cols)
rows_after, cols_after = df_clean.shape

print()
print(f"Dropped {len(drop_cols)} columns with null rate ≥ {NULL_DROP_THRESHOLD:.0%}")
print("Dropped columns:")
for c in drop_cols:
    print(f"  - {c}: {null_rate[c]:.1%}")
print()
print(f"Shape before: {rows_before:,} × {cols_before:,}")
print(f"Shape after:  {rows_after:,} × {cols_after:,}")

## 3. Numeric vs categorical features

**Rule for this guided session (competition AMEX contract + dtype):**

- Exclude `customer_ID` and `target` from feature lists.
- **Categorical features** = union of:
  1. Official AMEX prepare-contract aggregates present in the frame: `{col}_mode` / `{col}_last` for  
     `B_30`, `B_38`, `D_114`, `D_116`, `D_117`, `D_120`, `D_126`, `D_63`, `D_64`, `D_66`, `D_68`  
     (even when stored as float codes — dtype alone is misleading on the book-sample mart).
  2. Any other remaining non-numeric dtypes (`object` / `string` / `category`).
- **Numeric features:** everything else among the feature columns.

We also print a **dtype-only** count for comparison (book-sample path often shows only ~2).

In [ ]:
def classify_features(frame: pd.DataFrame):
    cols = [c for c in frame.columns if c not in {ID_COL, TARGET}]
    dtype_num = set(frame[cols].select_dtypes(include="number").columns)
    dtype_cat = [c for c in cols if c not in dtype_num]
    contract_cat = sorted(c for c in cols if c in AMEX_CAT_FEATURE_NAMES)
    cats = sorted(set(dtype_cat) | set(contract_cat))
    nums = [c for c in cols if c not in cats]
    return cols, cats, nums, dtype_cat, contract_cat


# Inventory on raw loaded frame (before null drop) vs cleaned frame
_, cats_before, _, dtype_before, contract_before = classify_features(df)
feature_cols, categorical_features, numeric_features, dtype_categorical, contract_categorical = (
    classify_features(df_clean)
)
cats_dropped_by_null = [c for c in cats_before if c in drop_cols]

print("=== BEFORE ≥80% null drop ===")
print(f"Categorical (contract ∪ non-numeric): {len(cats_before)}")
print("List:", ", ".join(cats_before))
print(f"dtype-only non-numeric: {len(dtype_before)} → {dtype_before}")
print()
print(
    f"Categoricals dropped by null threshold: {len(cats_dropped_by_null)} → {cats_dropped_by_null}"
)
print()
print("=== AFTER ≥80% null drop ===")
print(f"Feature columns (excl. id/target): {len(feature_cols)}")
print(f"Categorical features (contract ∪ non-numeric dtype): {len(categorical_features)}")
print(f"  of which official AMEX *_mode/*_last present: {len(contract_categorical)}")
print(f"  of which dtype-only non-numeric: {len(dtype_categorical)}")
print(f"Numeric features (remainder): {len(numeric_features)}")
print()
print("Categorical list:")
for c in categorical_features:
    print(f"  - {c}  dtype={df_clean[c].dtype}")
print()
print("Numeric sample:", numeric_features[:12])
print()
print("Dtype value counts (feature columns only):")
print(df_clean[feature_cols].dtypes.value_counts().to_string())

## 4. Class imbalance (`target`)

`target = 1` → **bad** (default). `target = 0` → **good**.

Credit default datasets are usually **imbalanced** (far more goods than bads). We report counts, rates, and a simple bar chart so the imbalance is visible before any modeling.

In [ ]:
BAD, GOOD = 1, 0

counts = df_clean[TARGET].value_counts(dropna=False).sort_index()
n_good = int((df_clean[TARGET] == GOOD).sum())
n_bad = int((df_clean[TARGET] == BAD).sum())
n_total = n_good + n_bad
pct_good = n_good / n_total
pct_bad = n_bad / n_total

balance = pd.DataFrame(
    {
        "label": ["good (0)", "bad (1)"],
        "n": [n_good, n_bad],
        "pct": [pct_good, pct_bad],
    }
)
print(balance.to_string(index=False, formatters={"pct": "{:.2%}".format}))
print()
print(f"Imbalance ratio (good:bad) ≈ {n_good / max(n_bad, 1):.2f}:1")
print(f"Default / bad rate: {pct_bad:.2%}")

ax = counts.plot(kind="bar", title="target balance (0=good, 1=bad)", rot=0, figsize=(5, 3))
ax.set_xlabel("target")
ax.set_ylabel("count")

## Slice 1 checkpoint (verified — competition AMEX mart)

- **Source:** competition AMEX mart — `data/credit/application_mart/amex/application_mart.parquet` (rebuilt with `build-competition-mart`; #157 contract)
- **Shape before null drop:** 458,913 rows × 909 columns
- **Columns dropped (≥80% null):** 107 (includes categorical `D_66_mode`, `D_66_last`)
- **Shape after null drop:** 458,913 rows × 802 columns
- **Categoricals before null drop:** 22 (all official `{col}_mode`/`{col}_last`)
- **Categoricals after null drop:** 20 (lost `D_66_mode`, `D_66_last`); numeric features: 780
- **dtype-only non-numeric:** only `D_63_*` / `D_64_*` (strings); other official cats are float-coded `*_mode`/`*_last` — hence contract ∪ dtype rule
- **Class balance:** good (0) ≈ 340,085 (74.11%); bad (1) ≈ 118,828 (25.89%)

Continue below with **step 5** (univariate vs target).


## 5. Univariate association with `target`

Portfolio **baseline bad rate** ≈ class imbalance from step 4 (~26%). Here we compare each feature level / value band to that baseline.

### 5a. All remaining AMEX categoricals (~20)

For every official `{col}_mode` / `{col}_last` still present after the ≥80% null drop (`D_66_*` already gone), compute:

- bad rate by level (`mean(target)`)
- count `n` (flag small cells; ranking uses `n ≥ 500`)
- lift vs baseline

Sort / highlight levels with the highest and lowest bad rates.


In [ ]:
import matplotlib.pyplot as plt

baseline = float(df_clean[TARGET].mean())
print(f"Baseline bad rate: {baseline:.2%}  (n={len(df_clean):,})")
print(f"Categoricals in this pass: {len(categorical_features)}")
assert len(categorical_features) == 20, "Expected 20 cats after D_66 null drop on competition mart"

MIN_N_RANK = 500

cat_rows = []
for col in categorical_features:
    g = df_clean.groupby(col, dropna=False)[TARGET].agg(bad_rate="mean", n="count").reset_index()
    g["feature"] = col
    g["level"] = g[col].map(lambda v: "nan" if pd.isna(v) else str(v))
    g["lift_vs_baseline"] = g["bad_rate"] - baseline
    cat_rows.append(g[["feature", "level", "bad_rate", "n", "lift_vs_baseline"]])

cat_bad_rates = pd.concat(cat_rows, ignore_index=True)
rankable = cat_bad_rates[cat_bad_rates["n"] >= MIN_N_RANK]

print()
print(f"=== Highest bad-rate levels (n≥{MIN_N_RANK}) ===")
print(
    rankable.sort_values("bad_rate", ascending=False)
    .head(12)
    .to_string(
        index=False, formatters={"bad_rate": "{:.1%}".format, "lift_vs_baseline": "{:+.1%}".format}
    )
)
print()
print(f"=== Lowest bad-rate levels (n≥{MIN_N_RANK}) ===")
print(
    rankable.sort_values("bad_rate", ascending=True)
    .head(12)
    .to_string(
        index=False, formatters={"bad_rate": "{:.1%}".format, "lift_vs_baseline": "{:+.1%}".format}
    )
)

# Per-feature spread (max − min bad rate among levels with n≥200)
spreads = []
for col in categorical_features:
    sub = cat_bad_rates[(cat_bad_rates["feature"] == col) & (cat_bad_rates["n"] >= 200)]
    if sub.empty:
        continue
    hi = sub.loc[sub["bad_rate"].idxmax()]
    lo = sub.loc[sub["bad_rate"].idxmin()]
    spreads.append(
        {
            "feature": col,
            "hi_level": hi["level"],
            "hi_rate": hi["bad_rate"],
            "hi_n": int(hi["n"]),
            "lo_level": lo["level"],
            "lo_rate": lo["bad_rate"],
            "lo_n": int(lo["n"]),
            "spread": hi["bad_rate"] - lo["bad_rate"],
        }
    )
spread_df = pd.DataFrame(spreads).sort_values("spread", ascending=False)
print()
print("=== Per-feature spread (n≥200) ===")
print(
    spread_df.to_string(
        index=False,
        formatters={
            "hi_rate": "{:.1%}".format,
            "lo_rate": "{:.1%}".format,
            "spread": "{:.1%}".format,
        },
    )
)

# Faceted bad-rate bars for the 8 widest-spread categoricals
plot_feats = spread_df["feature"].head(8).tolist()
fig, axes = plt.subplots(2, 4, figsize=(16, 8), constrained_layout=True)
for ax, col in zip(axes.ravel(), plot_feats, strict=True):
    sub = cat_bad_rates[cat_bad_rates["feature"] == col].sort_values("bad_rate")
    colors = ["#c0392b" if r > baseline else "#2980b9" for r in sub["bad_rate"]]
    ax.barh(sub["level"].tolist(), sub["bad_rate"].tolist(), color=colors)
    ax.axvline(baseline, color="black", ls="--", lw=1)
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("bad rate")
    ax.set_xlim(0, max(0.6, float(sub["bad_rate"].max()) * 1.05))
fig.suptitle(f"Categorical bad rate by level (baseline={baseline:.1%}; red>baseline)", fontsize=12)
plt.show()

### 5b. A few numerics only (not all 780)

Screen numerics with a simple association (`|corr(feature, target)|`), keep the top ~8, then show:

1. distribution overlap (good vs bad), clipped to the 1–99% range for readability
2. decile-binned bad rate vs baseline

This is a light pass — not a full correlation matrix or outlier study (those are later steps).


In [ ]:
# Screen numerics by |corr| with target (pairwise complete; light pass — top ~8 only)
y = df_clean[TARGET]
num_frame = df_clean[numeric_features].apply(pd.to_numeric, errors="coerce")
corr_s = num_frame.corrwith(y).dropna()
num_assoc = (
    corr_s.rename("corr")
    .to_frame()
    .assign(abs_corr=lambda d: d["corr"].abs())
    .sort_values("abs_corr", ascending=False)
)
# mean difference bad vs good for the shortlist only
top_n = 8
top_feats = num_assoc.head(top_n).index.tolist()
rows = []
for col in top_feats:
    x = num_frame[col]
    mask = x.notna()
    rows.append(
        {
            "feature": col,
            "corr": float(num_assoc.loc[col, "corr"]),
            "abs_corr": float(num_assoc.loc[col, "abs_corr"]),
            "mean_bad": float(x[mask & (y == BAD)].mean()),
            "mean_good": float(x[mask & (y == GOOD)].mean()),
        }
    )
top_nums = pd.DataFrame(rows)
top_nums["mean_diff"] = top_nums["mean_bad"] - top_nums["mean_good"]
print(f"Screened {len(num_assoc):,} numerics; showing top {top_n} by |corr|")
print(
    top_nums.to_string(
        index=False,
        formatters={
            "corr": "{:+.3f}".format,
            "abs_corr": "{:.3f}".format,
            "mean_bad": "{:.3f}".format,
            "mean_good": "{:.3f}".format,
            "mean_diff": "{:+.3f}".format,
        },
    )
)

# Overlap histograms for top 4
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
for ax, col in zip(axes.ravel(), top_feats[:4], strict=True):
    s = df_clean[[TARGET, col]].dropna()
    lo, hi = s[col].quantile([0.01, 0.99])
    s = s[(s[col] >= lo) & (s[col] <= hi)]
    ax.hist(
        s.loc[s[TARGET] == GOOD, col],
        bins=40,
        density=True,
        alpha=0.55,
        label="good(0)",
        color="#2980b9",
    )
    ax.hist(
        s.loc[s[TARGET] == BAD, col],
        bins=40,
        density=True,
        alpha=0.55,
        label="bad(1)",
        color="#c0392b",
    )
    r = float(top_nums.loc[top_nums["feature"] == col, "corr"].iloc[0])
    ax.set_title(f"{col}  corr={r:+.3f}")
    ax.legend(fontsize=8)
fig.suptitle("Numeric overlap: good vs bad (clipped 1–99%)")
plt.show()

# Decile-binned bad rate for top 6
fig, axes = plt.subplots(2, 3, figsize=(14, 8), constrained_layout=True)
for ax, col in zip(axes.ravel(), top_feats[:6], strict=True):
    s = df_clean[[TARGET, col]].dropna().copy()
    try:
        s["bin"] = pd.qcut(s[col], q=10, duplicates="drop")
    except ValueError:
        s["bin"] = pd.cut(s[col], bins=10)
    br = s.groupby("bin", observed=True)[TARGET].mean()
    ax.plot(range(len(br)), br.to_numpy(), marker="o", color="#c0392b")
    ax.axhline(baseline, color="black", ls="--", lw=1)
    ax.set_xticks(range(len(br)))
    ax.set_xticklabels([str(i + 1) for i in range(len(br))], fontsize=7)
    ax.set_ylim(0, max(0.55, float(br.max()) * 1.1))
    ax.set_title(col, fontsize=9)
    ax.set_xlabel("decile")
    ax.set_ylabel("bad rate")
fig.suptitle(f"Binned bad rate for top |corr| numerics (baseline={baseline:.1%})")
plt.show()

## Step 5 summary (verified local run — competition AMEX mart)

**Baseline bad rate:** 25.89% (118,828 / 458,913)

### Top categorical findings (n≥500)

| Feature | Level | Bad rate | n | vs baseline |
| --- | --- | --- | --- | --- |
| `B_38_mode` | 4.0 | **72.2%** | 27,001 | +46.3 pp |
| `B_38_last` | 4.0 | **72.0%** | 25,901 | +46.1 pp |
| `D_116_last` | 1.0 | **70.7%** | 1,332 | +44.8 pp (small cell — watch) |
| `B_30_last` | 1.0 | **66.2%** | 71,127 | +40.3 pp |
| `B_38_last` | 2.0 | **5.1%** | 152,555 | −20.7 pp (strong good) |

`B_38_*` dominates both tails; `B_30=1` is a large high-risk pocket; `D_116=1` is extreme but thin.

### Numerics inspected (top `|corr|`)

| Feature | corr | One-line takeaway |
| --- | --- | --- |
| `P_2_last` | −0.667 | Higher payment-like score → much lower bad rate; strongest single numeric |
| `P_2_min` / `P_2_mean` / `P_2_max` | −0.64 / −0.63 / −0.59 | Same family; bad accounts sit lower across the history |
| `D_48_last` / `D_48_mean` | +0.61 / +0.58 | Higher delinquency-like values → higher bad rate (mirror of P_2) |
| `R_1_std` / `B_2_last` | +0.57 / −0.56 | Volatility / balance-like signal also separates, slightly weaker |

### Stop

**Stop here — await Jorge's valuation before step 6** (correlation / outliers / modeling).
